<a href="https://colab.research.google.com/github/DeepakSaini01/FlyRank-AI-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!git clone https://github.com/DeepakSaini01/FlyRank-AI-internship.git

fatal: destination path 'ML-WEEK-1' already exists and is not an empty directory.


In [14]:
import os, warnings
import pandas as pd, numpy as np
import pyarrow.dataset as ds
from huggingface_hub import login, HfApi
import matplotlib.pyplot as plt, seaborn as sns

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

# ------------------------------------------------------------
# 0️⃣ Auth
# ------------------------------------------------------------
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

REPO = "FlyRank/internship-warehouse"
FOLDER = "fact_content_daily_performance"
TARGET_MONTH = "2026-03"

# ------------------------------------------------------------
# 1️⃣ List files in the target folder (fast, metadata only)
# ------------------------------------------------------------
api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files(REPO, repo_type="dataset")
daily_files = [f for f in all_files if f.startswith(f"{FOLDER}/") and f.endswith(".parquet")]
print(f"Found {len(daily_files)} parquet files in {FOLDER}/")

if not daily_files:
    raise FileNotFoundError(f"No parquet files found under {FOLDER}/ — check the folder name")

# ------------------------------------------------------------
# 2️⃣ Best-effort filename pruning — only helps if filenames
#    actually encode the month; otherwise falls back safely
#    to reading everything and filtering by content.
# ------------------------------------------------------------
narrowed = [f for f in daily_files if TARGET_MONTH in f]
files_to_read = narrowed if narrowed else daily_files
print(f"Reading {len(files_to_read)} file(s) "
      f"({'filename-pruned to ' + TARGET_MONTH if narrowed else 'no matching filename pattern — reading all, will filter by content'})")

paths = [f"hf://datasets/{REPO}/{f}" for f in files_to_read]

# ------------------------------------------------------------
# 3️⃣ Load with column pushdown (only pulls the columns we need)
# ------------------------------------------------------------
cols = ["content_hash_id", "client_hash_id", "report_date",
        "gsc_impressions", "gsc_clicks", "gsc_avg_position"]

dataset = ds.dataset(paths, format="parquet")
table = dataset.to_table(columns=cols)
df = table.to_pandas()

# ------------------------------------------------------------
# 4️⃣ Filter to target month
# ------------------------------------------------------------
df = df[df["report_date"].astype(str).str.startswith(TARGET_MONTH)].copy()

# ------------------------------------------------------------
# 5️⃣ Rename columns
# ------------------------------------------------------------
df = df.rename(columns={
    "content_hash_id": "content_id",
    "client_hash_id":  "client_id",
    "gsc_impressions": "impressions_90d",
    "gsc_clicks":      "clicks_90d",
    "gsc_avg_position":"avg_position",
})

# ------------------------------------------------------------
# 6️⃣ Derive month, drop raw date
# ------------------------------------------------------------
df["month"] = pd.to_datetime(df["report_date"]).dt.strftime("%Y-%m")
df.drop(columns=["report_date"], inplace=True)

# ------------------------------------------------------------
# 7️⃣ Compute CTR
# ------------------------------------------------------------
df["ctr"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    np.nan,
)
df["ctr"] = df["ctr"].fillna(df["ctr"].median())

# ------------------------------------------------------------
# 8️⃣ Sanity check
# ------------------------------------------------------------
print("\n✅ Daily slice ready")
print(" • Unique month(s) :", df["month"].unique())
print(" • Row count       :", df.shape[0])
print(" • Columns         :", df.columns.tolist())

Found 18 parquet files in fact_content_daily_performance/
Reading 1 file(s) (filename-pruned to 2026-03)


ArrowInvalid: Expected a local filesystem path, got a URI: 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content page

Table used = flyrank_monthly

Time window = March 2026 (2026‑03)

Target = binary declining label (trend_direction == "down")

Exclude = internal ML‑system signals (provider_used, model_used, ai_sessions_90d)

In [ ]:
# ------------------------------------------------------------
# 1️⃣ Verify that “one row = one content page” for March 2026
# ------------------------------------------------------------

# Keep only the development month (mid‑panel month)
march_mask = df["month"] == "2026-03"
march_df   = df.loc[march_mask].copy()

print("\n📅 Selected month:", march_df["month"].unique())
print("🧮 Number of rows for March 2026:", march_df.shape[0])

# Verify that the grain is (content_id, month)
grain_check = (
    march_df.groupby(["content_id", "month"])
    .size()
    .reset_index(name="rows_per_key")
)

print("\n🔎 Distinct (content_id, month) pairs:", grain_check.shape[0])
print("🔎 Any duplicate rows per key?", (grain_check["rows_per_key"] > 1).any())

# Show a few example rows so we can see the schema
display(march_df.head())

NameError: name 'df' is not defined

## 2. Fields: feature / label / context / excluded

| Bucket      | Columns (why they belong)                                                                                                                                 |
|------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Feature**| `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `competition_level` – observable at the decision month and strong signals of page health.         |
| **Label**  | `trend_direction` → binary **`is_declining_label = (trend_direction == "down")`**. This proxy tells whether the page will decline in the **next month** (April 2026). |
| **Context**| `content_type`, `main_intent`, `client_id` – useful for downstream segment analysis (e.g., “how does competition affect different intents?”).               |
| **Excluded**| `provider_used`, `model_used`, `ai_sessions_90d` – these are internal ML‑system signals that are not part of the organic search‑console data and would give an unfair advantage if used. |


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check schema and field null counts


## 3. Verify it with queries (grain, counts, missing values, windows)



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification checks
# ------------------------------------------------------------
# 1️⃣ Grain check: (content_id, month) must be unique
# ------------------------------------------------------------
grain_check = (
    march_df.groupby(["content_id", "month"])
    .size()
    .reset_index(name="rows_per_key")
)

unique_pairs = grain_check.shape[0]
duplicates   = grain_check["rows_per_key"].gt(1).sum()

print("🔎 Grain verification")
print(f" • Distinct (content_id, month) pairs : {unique_pairs:,}")
print(f" • Duplicate rows for any pair        : {duplicates}")

# Assertion (optional – will raise if the contract is broken)
assert duplicates == 0, "Grain violation: duplicate rows found!"

# ------------------------------------------------------------
# 2️⃣ Row counts & window span
# ------------------------------------------------------------
total_rows      = df.shape[0]                     # full table (all months)
march_rows      = march_df.shape[0]                # rows for the chosen month
month_range     = (df["month"].min(), df["month"].max())

print("\n📈 Row‑count summary")
print(f" • Total rows in the whole dataset   : {total_rows:,}")
print(f" • Rows for the target month (2026‑03): {march_rows:,}")

print("\n🗓️ Temporal window covered by the slice")
print(f" • Earliest month : {month_range[0]}")
print(f" • Latest month   : {month_range[1]}")

# ------------------------------------------------------------
# 3️⃣ Missing‑value overview (per column)
# ------------------------------------------------------------
missing_counts = march_df.isna().sum()
missing_pct   = (missing_counts / march_df.shape[0] * 100).round(2)

missing_df = pd.DataFrame({
    "missing_vals": missing_counts,
    "pct_missing": missing_pct
}).sort_values("missing_vals", ascending=False)

print("\n❓ Missing‑value summary (target month only)")
display(missing_df.head(15))   # show the 15 columns with most missing data

# ------------------------------------------------------------
# 4️⃣ Window sanity check – confirm that the month column
#     really spans the expected period (e.g. March 2026 only)
# ------------------------------------------------------------
# For the contract we expect *exactly* March 2026.
unique_months = march_df["month"].unique()
print("\n⏳ Window sanity")
print(f" • Unique months present in the slice : {list(unique_months)}")
assert set(unique_months) == {"2026-03"}, "Unexpected month(s) found!"

# If you want to see the min / max of any numeric column as a quick “range” check:
numeric_cols = march_df.select_dtypes(include="number").columns
range_summary = march_df[numeric_cols].agg(["min", "max"]).T
print("\n📏 Numeric column ranges (first 5 rows shown)")
display(range_summary.head())

## 4. Data limits

* **No true temporal granularity** – the table only carries a `month` column, not a day‑level `date`.  
  → We cannot build day‑by‑day trend lines or rolling‑window forecasts.

* **Unbalanced historical coverage** – early months (2025‑early‑2026) have far fewer rows because the ingestion pipeline was still being built.

  → Any long‑term trend must be interpreted with caution.
* **Search‑Console‑only early rows** – for the first few months only Search Console signals are present; other columns (e.g., `ai_traffic_pct`) are missing.

  → Feature completeness varies over time.
  
* **Window overlap** – because `month` aggregates all activity for a calendar month, a single `content_id` can appear in multiple months with different signal values, preventing a true “single‑snapshot” view.  
  → The slice cannot answer “what was the exact state on day X”.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature distributions and zero-value constraints


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.